# 03b — Model improvement experiments (Option B)

Tests four hypotheses raised while reviewing the SHAP results in `03_models.ipynb`, against the actual CatBoost model (Option B, time split). This notebook is deliberately separate from `03_models.ipynb` — it is exploratory, not part of the stable pipeline, and meant to be run on its own branch.

**Result up front, not buried at the end:** of the four things tested, only one held up. The other three are reported anyway, with the numbers that ruled them out — a suggestion that doesn't survive contact with the data is still worth documenting, so it isn't tried again later without remembering why it didn't work.

| # | Hypothesis | Outcome |
|---|---|---|
| 1 | Mental-health features hide multicollinearity beyond what full-dataset VIF caught | **Not confirmed** — max VIF 3.64 within the block |
| 2 | Test/Val gap is partly a COVID-period effect, not just time-distance decay | **Confirmed** |
| 3 | Dropping lowest-importance features improves generalization | **Not confirmed** — Val and Test R² both dropped |
| 4 | The CatBoost `l2_leaf_reg` optimum lies beyond the grid already searched | **Not confirmed** — GridSearchCV re-selected the same value |

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostRegressor

from src import (
    ID_COLS, TARGET, HEALTH_RELATED_FEATURES, build_predictor_list, temporal_split,
    compute_vif, train_model, evaluate_model, metrics_by_period,
)

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
np.random.seed(42)

df_development = pd.read_csv("../data/processed/df_development.csv")
predictor_features = build_predictor_list(df_development, ID_COLS, TARGET)
print(f"df_development: {df_development.shape[0]} rows | {len(predictor_features)} predictors")

## Shared setup (Option B — same split as 03_models.ipynb)

Rebuilds the same train/test/val split and scaling used in `03_models.ipynb`, so every variant below is compared on identical data. `scale_split()` is a small local helper — not promoted to `src/` because it only exists to let this notebook re-scale different feature subsets (baseline vs trimmed) without repeating the fit/transform boilerplate each time.

In [ ]:
df_train_B, df_test_B, df_val_B, train_years, test_years, val_years = temporal_split(df_development)
y_train_B, y_test_B, y_val_B = df_train_B[TARGET], df_test_B[TARGET], df_val_B[TARGET]


def scale_split(feature_list):
    scaler = RobustScaler()
    X_train = pd.DataFrame(scaler.fit_transform(df_train_B[feature_list]), columns=feature_list, index=df_train_B.index)
    X_test = pd.DataFrame(scaler.transform(df_test_B[feature_list]), columns=feature_list, index=df_test_B.index)
    X_val = pd.DataFrame(scaler.transform(df_val_B[feature_list]), columns=feature_list, index=df_val_B.index)
    return X_train, X_test, X_val


X_train_B, X_test_B, X_val_B = scale_split(predictor_features)

# Baseline: exact hyperparameters GridSearchCV selected for CatBoost in 03_models.ipynb,
# refit here directly (no search) purely as the reference point for the comparisons below.
baseline_model = CatBoostRegressor(depth=4, iterations=400, l2_leaf_reg=7, learning_rate=0.1,
                                    random_state=42, verbose=0)
trained_baseline = {
    "name": "CatBoost (baseline)",
    "best_estimator": baseline_model.fit(X_train_B, y_train_B),
    "best_params": {"depth": 4, "iterations": 400, "l2_leaf_reg": 7, "learning_rate": 0.1},
    "cv_rmse": None, "time_s": None,
}
eval_baseline_test = evaluate_model(trained_baseline, X_test_B, y_test_B, "Test")
eval_baseline_val = evaluate_model(trained_baseline, X_val_B, y_val_B, "Val")

## Hypothesis 1 — Hidden multicollinearity within the mental-health block

`02_eda.ipynb`'s VIF check ran on the full predictor set and only flagged `Eating disorders` (VIF 12.3, already dropped). The concern here was narrower: mental-health prevalence causes come from the same underlying study (GBD) and might share enough variance with *each other* specifically to distort SHAP's attribution, even without triggering the full-dataset threshold. Re-running VIF restricted to just this block checks that directly.

In [ ]:
mental_health_features = [f for f in HEALTH_RELATED_FEATURES if f != TARGET]
vif_mental_health = compute_vif(df_development, mental_health_features)
vif_mental_health

**Result: not confirmed.** The highest VIF within the block is 3.64 (`Attention-deficit/hyperactivity disorder`) — well under the moderate-concern threshold of 5. There is no meaningful hidden collinearity here; `Alcohol use disorders` dominating the SHAP ranking is not an artifact of correlated features splitting credit unevenly. It needs a different explanation than multicollinearity.

## Hypothesis 2 — Is the Test/Val gap a COVID-period effect?

Splits Option B's Validation years (2018–2021) into Pre-COVID (2018–2019) and COVID (2020–2021), and computes metrics separately for each, using `metrics_by_period()`.

In [ ]:
periods = {"Pre-COVID (2018-2019)": [2018, 2019], "COVID (2020-2021)": [2020, 2021]}

period_metrics = metrics_by_period(
    eval_baseline_val["actuals"], eval_baseline_val["predictions"],
    df_val_B["Year"].reset_index(drop=True), periods,
)
period_metrics

**Result: confirmed.** R² drops from 0.72 (Pre-COVID) to 0.47 (COVID) within the same Validation set, using the same model. A meaningful share of the overall Val R² (0.605) reported in `03_models.ipynb` is attributable to a two-year structural shock the model — trained on 2000–2014 — had no way to anticipate, not to a generic "the further in time, the worse" decay. Worth stating explicitly in the thesis discussion rather than folding it into a single Val R² number.

## Hypothesis 3 — Dropping the lowest-importance features improves generalization

Drops the three lowest-ranked features from both the SHAP and CatBoost native importance rankings computed in `03_models.ipynb` (`Health expenditure (% GDP)`, `Unemployment rate (%)`, `Conduct disorder`) and retrains with identical hyperparameters, to isolate the effect of the feature set alone.

In [ ]:
LOW_IMPORTANCE_FEATURES = ["Health expenditure (% GDP)", "Unemployment rate (%)", "Conduct disorder"]
trimmed_features = build_predictor_list(df_development, ID_COLS, TARGET, extra_exclude=LOW_IMPORTANCE_FEATURES)
print(f"Predictors: {len(predictor_features)} -> {len(trimmed_features)}")

X_train_trim, X_test_trim, X_val_trim = scale_split(trimmed_features)

trimmed_model = CatBoostRegressor(depth=4, iterations=400, l2_leaf_reg=7, learning_rate=0.1,
                                   random_state=42, verbose=0)
trained_trimmed = {
    "name": "CatBoost (trimmed)",
    "best_estimator": trimmed_model.fit(X_train_trim, y_train_B),
    "best_params": {}, "cv_rmse": None, "time_s": None,
}
eval_trimmed_test = evaluate_model(trained_trimmed, X_test_trim, y_test_B, "Test")
eval_trimmed_val = evaluate_model(trained_trimmed, X_val_trim, y_val_B, "Val")

print(f"\nBaseline  — Test R²: {eval_baseline_test['r2']} | Val R²: {eval_baseline_val['r2']}")
print(f"Trimmed   — Test R²: {eval_trimmed_test['r2']} | Val R²: {eval_trimmed_val['r2']}")

**Result: not confirmed — it makes things slightly worse.** Test R² drops from 0.875 to 0.848, Val R² from 0.605 to 0.577. [Seguro] With 594 rows and 18 predictors, this dataset is not in the regime where CatBoost benefits from feature pruning — the "noise" hypothesis behind this experiment doesn't hold. The three dropped features were carrying some real signal despite their low individual SHAP ranking, likely through interactions rather than main effects. Keep the full feature set.

## Hypothesis 4 — Is the `l2_leaf_reg` optimum beyond the searched range?

The original grid in `src/models.py` searches `l2_leaf_reg` in `[3, 7]`, and GridSearchCV picked `7` — the upper bound. That pattern (best value at the edge of the grid) is usually a sign the true optimum lies further out. Re-runs GridSearchCV with the range extended to `[3, 7, 10, 15]`, everything else identical, to check.

Uses a plain dict literal here rather than editing `src/models.py`'s `param_grids["CatBoost"]` directly — this keeps the experiment self-contained on this branch without touching the grid `03_models.ipynb` already ran and committed results against.

In [ ]:
cv_B = TimeSeriesSplit(n_splits=5)
extended_grid = {
    "iterations": [200, 400],
    "depth": [3, 4, 5],
    "learning_rate": [0.05, 0.1],
    "l2_leaf_reg": [3, 7, 10, 15],  # extended from [3, 7]
}

trained_extended = train_model(
    "CatBoost (extended grid)", CatBoostRegressor(random_state=42, verbose=0),
    extended_grid, X_train_B, y_train_B, cv_B,
)
eval_extended_test = evaluate_model(trained_extended, X_test_B, y_test_B, "Test")
eval_extended_val = evaluate_model(trained_extended, X_val_B, y_val_B, "Val")

**Result: not confirmed.** GridSearchCV re-selected `l2_leaf_reg=7` — the same value found with the smaller grid — and Test/Val R² are unchanged. The apparent "edge of the grid" pattern was coincidental, not a sign of an unexplored better region. No change needed here.

## Summary

| Variant | Test R² | Val R² | Change vs baseline |
|---|---|---|---|
| Baseline (as in `03_models.ipynb`) | 0.875 | 0.605 | — |
| Trimmed features | 0.848 | 0.577 | Worse |
| Extended `l2_leaf_reg` grid | 0.875 | 0.605 | No change |

Out of the four things tested, the only one worth carrying back into the thesis write-up is the COVID-period breakdown (Hypothesis 2) — it's a genuine, evidenced qualification of the Option B results, not a model change. The other three are negative results: worth keeping in this notebook as a record of what was tried and ruled out, but none of them justify changing `src/models.py` or `03_models.ipynb`.